<a href="https://colab.research.google.com/github/KDB4926/5th-diet-ai/blob/main/Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QA 전처리

In [ ]:
"""
Team DIET 법률 AI 데이터 전처리 파이프라인 (최종 완성본)
- 기능 1: QA / SUM 역할에 따라 파일 자동 필터링
- 기능 2: 구글 드라이브 '바로가기' 경로 완벽 대응
- 기능 3: 20,000개 대량 처리 대비 (안정성 강화)
"""

# ========================================================
# 1. 라이브러리 설치 및 설정
# ========================================================
!pip install -q -U openai pandas tqdm jsonlines

import openai
import json
import jsonlines
import os
from tqdm import tqdm
from google.colab import drive, userdata
import time

# 구글 드라이브 연결
drive.mount('/content/drive')

# ========================================================
# 2. OpenAI API 설정 (필수)
# ========================================================
print("\n🔑 OpenAI API 설정을 확인합니다...")

api_key = input("👉 OpenAI API Key를 입력하세요 (sk-...): ")
api_key = api_key.strip()

client = openai.OpenAI(api_key=api_key)
TARGET_MODEL = "gpt-4o-mini"

# ========================================================
# 3. 프롬프트 템플릿 (수정 금지)
# ========================================================
QA_PROMPT_TEMPLATE = """
너는 'Team DIET'의 수석 법률 분석가야.
아래 판결문 내용을 심층 분석하여, 실무 공무원을 위한 고품질 '법률 질의응답' 1세트를 작성해줘.

[판결문 전문]
{context}

[작성 가이드]
1. 질문(instruction):
   - '판단 기준'이나 '법적 해석'을 묻는 구체적인 실무 질문일 것.
2. 답변(output):
   - '결론(두괄식) → 법령/판례 근거 → 사안의 포섭(적용)'의 3단 논법 필수.
   - 논리적이고 명확한 법률 전문가 말투 사용.

[출력 형식 (JSON)]
{{
  "instruction": "작성된 질문",
  "input": "",
  "output": "작성된 답변"
}}
"""

# ========================================================
# 4. 데이터 파싱 및 변환 함수
# ========================================================
def parse_legal_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        content = ""
        # 데이터 구조에 따라 본문 추출
        if 'info' in data and 'Precedent' in data['info']:
            content = data['info']['Precedent'].get('PrecedentContent', '')

        if not content or len(content) < 50:
            return None
        return content
    except:
        return None

def convert_with_gpt(context_text):
    try:
        response = client.chat.completions.create(
            model=TARGET_MODEL,
            messages=[
                {"role": "system", "content": "너는 법률 데이터를 JSON 형식으로 변환하는 전문가야."},
                {"role": "user", "content": QA_PROMPT_TEMPLATE.format(context=context_text[:15000])}
            ],
            response_format={"type": "json_object"},
            temperature=0.3
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        # 에러 발생 시 로그 출력 (디버깅용)
        # print(f"\n❌ [오류 발생] {e}")
        return None

# ========================================================
# 5. 메인 실행 로직 (필터링 기능 포함)
# ========================================================
def process_directory(input_dir, output_file, keyword, max_files=None):
    target_files = []

    print(f"🔍 폴더를 스캔하며 '{keyword}' 파일만 찾는 중...")

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".json"):
                full_path = os.path.join(root, file)
                # 경로(폴더명) 또는 파일명에 내 역할(keyword)이 포함된 것만 선택
                if keyword in full_path:
                    target_files.append(full_path)

    if len(target_files) == 0:
        print(f"❌ '{keyword}' 관련 파일을 하나도 못 찾았습니다! 경로를 다시 확인해주세요.")
        return 0

    # 갯수 제한 적용
    final_files = target_files[:max_files] if max_files else target_files
    print(f"🧪 처리 대상({keyword}): {len(final_files)}개 (전체 {len(target_files)}개 중)")

    success_count = 0
    with jsonlines.open(output_file, mode='w') as writer:
        for file_path in tqdm(final_files, desc=f"GPT가 {keyword} 데이터 처리 중"):
            context = parse_legal_json(file_path)
            if not context: continue

            qa_pair = convert_with_gpt(context)
            if qa_pair:
                writer.write(qa_pair)
                success_count += 1

    return success_count

if __name__ == "__main__":

    # ---------------------------------------------------------
    # 👇 [설정 1] 본인의 역할에 맞게 수정하세요! ("QA" 또는 "SUM")
    # ---------------------------------------------------------
    MY_ROLE = "QA"
    # ---------------------------------------------------------

    # ---------------------------------------------------------
    # 👇 [설정 2] 처리할 파일 개수 (테스트용 100개 -> 실전용 20000개)
    # ---------------------------------------------------------
    TEST_LIMIT = 100
    # ---------------------------------------------------------

    # [경로 수정 완료] 내 드라이브 바로 밑에 있는 폴더 경로
    TRAIN_DIR = "/content/drive/MyDrive/02.라벨링데이터"

    OUTPUT_DIR = "/content/drive/MyDrive/legal_output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 결과 파일 이름 (역할에 따라 자동 변경)
    TRAIN_OUTPUT = f"{OUTPUT_DIR}/train_data_{MY_ROLE}.jsonl"

    if os.path.exists(TRAIN_DIR):
        print(f"🚀 Team DIET 전처리 엔진 시작! (담당: {MY_ROLE})")
        print(f"📂 데이터 읽는 곳: {TRAIN_DIR}")
        print(f"💾 결과 저장 곳: {TRAIN_OUTPUT}\n")

        count = process_directory(TRAIN_DIR, TRAIN_OUTPUT, keyword=MY_ROLE, max_files=TEST_LIMIT)

        print(f"\n✅ 작업 완료! 총 {count}개 생성됨.")
    else:
        print(f"❌ 오류: '{TRAIN_DIR}' 폴더가 없습니다. 구글 드라이브 바로가기가 잘 되었는지 확인해주세요.")

---

# SUM 전처리

In [ ]:
"""
Team DIET 법률 AI 데이터 전처리 파이프라인 (최종 완성본)
- 기능 1: QA / SUM 역할에 따라 파일 자동 필터링
- 기능 2: 구글 드라이브 '바로가기' 경로 완벽 대응
- 기능 3: 20,000개 대량 처리 대비 (안정성 강화)
"""

# ========================================================
# 1. 라이브러리 설치 및 설정
# ========================================================
!pip install -q -U openai pandas tqdm jsonlines

import openai
import json
import jsonlines
import os
from tqdm import tqdm
from google.colab import drive, userdata
import time

# 구글 드라이브 연결
drive.mount('/content/drive')

# ========================================================
# 2. OpenAI API 설정 (필수)
# ========================================================
print("\n🔑 OpenAI API 설정을 확인합니다...")

api_key = input("👉 OpenAI API Key를 입력하세요 (sk-...): ")
api_key = api_key.strip()

client = openai.OpenAI(api_key=api_key)
TARGET_MODEL = "gpt-4o-mini"

# ========================================================
# 3. 프롬프트 템플릿 (수정 금지)
# ========================================================
QA_PROMPT_TEMPLATE = """
너는 'Team DIET'의 수석 법률 분석가야.
아래 판결문 내용을 심층 분석하여, 실무 공무원을 위한 고품질 '법률 질의응답' 1세트를 작성해줘.

[판결문 전문]
{context}

[작성 가이드]
1. 질문(instruction):
   - '판단 기준'이나 '법적 해석'을 묻는 구체적인 실무 질문일 것.
2. 답변(output):
   - '결론(두괄식) → 법령/판례 근거 → 사안의 포섭(적용)'의 3단 논법 필수.
   - 논리적이고 명확한 법률 전문가 말투 사용.

[출력 형식 (JSON)]
{{
  "instruction": "작성된 질문",
  "input": "",
  "output": "작성된 답변"
}}
"""

# ========================================================
# 4. 데이터 파싱 및 변환 함수
# ========================================================
def parse_legal_json(file_path):
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        content = ""
        # 데이터 구조에 따라 본문 추출
        if 'info' in data and 'Precedent' in data['info']:
            content = data['info']['Precedent'].get('PrecedentContent', '')

        if not content or len(content) < 50:
            return None
        return content
    except:
        return None

def convert_with_gpt(context_text):
    try:
        response = client.chat.completions.create(
            model=TARGET_MODEL,
            messages=[
                {"role": "system", "content": "너는 법률 데이터를 JSON 형식으로 변환하는 전문가야."},
                {"role": "user", "content": QA_PROMPT_TEMPLATE.format(context=context_text[:15000])}
            ],
            response_format={"type": "json_object"},
            temperature=0.3
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        # 에러 발생 시 로그 출력 (디버깅용)
        # print(f"\n❌ [오류 발생] {e}")
        return None

# ========================================================
# 5. 메인 실행 로직 (필터링 기능 포함)
# ========================================================
def process_directory(input_dir, output_file, keyword, max_files=None):
    target_files = []

    print(f"🔍 폴더를 스캔하며 '{keyword}' 파일만 찾는 중...")

    for root, dirs, files in os.walk(input_dir):
        for file in files:
            if file.endswith(".json"):
                full_path = os.path.join(root, file)
                # 경로(폴더명) 또는 파일명에 내 역할(keyword)이 포함된 것만 선택
                if keyword in full_path:
                    target_files.append(full_path)

    if len(target_files) == 0:
        print(f"❌ '{keyword}' 관련 파일을 하나도 못 찾았습니다! 경로를 다시 확인해주세요.")
        return 0

    # 갯수 제한 적용
    final_files = target_files[:max_files] if max_files else target_files
    print(f"🧪 처리 대상({keyword}): {len(final_files)}개 (전체 {len(target_files)}개 중)")

    success_count = 0
    with jsonlines.open(output_file, mode='w') as writer:
        for file_path in tqdm(final_files, desc=f"GPT가 {keyword} 데이터 처리 중"):
            context = parse_legal_json(file_path)
            if not context: continue

            qa_pair = convert_with_gpt(context)
            if qa_pair:
                writer.write(qa_pair)
                success_count += 1

    return success_count

if __name__ == "__main__":

    # ---------------------------------------------------------
    # 👇 [설정 1] 본인의 역할에 맞게 수정하세요! ("QA" 또는 "SUM")
    # ---------------------------------------------------------
    MY_ROLE = "SUM"
    # ---------------------------------------------------------

    # ---------------------------------------------------------
    # 👇 [설정 2] 처리할 파일 개수 (테스트용 100개 -> 실전용 20000개)
    # ---------------------------------------------------------
    TEST_LIMIT = 100
    # ---------------------------------------------------------

    # [경로 수정 완료] 내 드라이브 바로 밑에 있는 폴더 경로
    TRAIN_DIR = "/content/drive/MyDrive/02.라벨링데이터"

    OUTPUT_DIR = "/content/drive/MyDrive/legal_output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 결과 파일 이름 (역할에 따라 자동 변경)
    TRAIN_OUTPUT = f"{OUTPUT_DIR}/train_data_{MY_ROLE}.jsonl"

    if os.path.exists(TRAIN_DIR):
        print(f"🚀 Team DIET 전처리 엔진 시작! (담당: {MY_ROLE})")
        print(f"📂 데이터 읽는 곳: {TRAIN_DIR}")
        print(f"💾 결과 저장 곳: {TRAIN_OUTPUT}\n")

        count = process_directory(TRAIN_DIR, TRAIN_OUTPUT, keyword=MY_ROLE, max_files=TEST_LIMIT)

        print(f"\n✅ 작업 완료! 총 {count}개 생성됨.")
    else:
        print(f"❌ 오류: '{TRAIN_DIR}' 폴더가 없습니다. 구글 드라이브 바로가기가 잘 되었는지 확인해주세요.")